# Passive compliance — guitar

How the hand's **torsional (joint-space) spring stiffness** shapes the interaction
when the closed fingers drag across the guitar strings.  Audio intensity is captured
externally via the camera microphone.  This notebook shows the **joint angle** and
**torque** state logged by the control loop during each sweep, as a sanity check
that the fingers were correctly closed and the springs were active.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('../../'))
from hand_config import (
    TORSIONAL_SPRINGS,
    GUITAR_N_RUNS             as N_RUNS,
    GUITAR_FINGER_CLOSED_POSE as FINGER_CLOSED_POSE,
    GUITAR_CLOSED_FINGERS     as CLOSED_FINGERS,
)

import shutil
if shutil.which('latex') is None:
    plt.rcParams['text.usetex'] = False

COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']
OUTPUT = os.path.join('outputs', 'guitar_playing_hand')
os.makedirs(OUTPUT, exist_ok=True)

# Motor indices for the four closed fingers (MCP, PIP)
# index=5,6  middle=7,8  ring=9,10  pinky=11,12
FINGER_MOTOR_IDX = {'index': [5, 6], 'middle': [7, 8], 'ring': [9, 10], 'pinky': [11, 12]}

def _synth(ktors):
    """Synthetic demo dataset (same schema as the real CSV)."""
    rng  = np.random.default_rng(int(ktors * 100))
    rows = []
    cols = ([f'q_{i}' for i in range(13)] +
            [f'qdot_{i}' for i in range(13)] +
            [f'tau_{i}' for i in range(13)])
    t = 0.0; dt = 0.02
    for run in range(1, N_RUNS + 1):
        for phase, length in [('sweep', 2.0), ('lift', 0.5), ('return', 0.5), ('descend', 0.5)]:
            n = int(length / dt)
            for _ in range(n):
                q   = np.zeros(13)
                tau = np.zeros(13)
                for f, idxs in FINGER_MOTOR_IDX.items():
                    for j in idxs:
                        q[j]   = FINGER_CLOSED_POSE[0] + rng.normal(0, 0.003)
                        tau[j] = ktors * 0.05 + rng.normal(0, 0.002)
                row = {'time_s': f'{t:.4f}', 'k_torsional': ktors, 'run': run, 'phase': phase}
                row.update({c: f'{v:.5f}' for c, v in zip(cols, list(q) + [0.0]*13 + list(tau))})
                rows.append(row); t += dt
        t += 0.3
    return pd.DataFrame(rows)

DEMO = False
def load(ktors):
    global DEMO
    path = os.path.join(OUTPUT, f'data_K{ktors:.2f}.csv')
    if os.path.isfile(path):
        return pd.read_csv(path)
    DEMO = True
    return _synth(ktors)

DATA = {k: load(k) for k in TORSIONAL_SPRINGS}
banner = ' DEMO (synthetic) data ' if DEMO else ' measured data '
print(f'Loaded{banner}|  K levels: {TORSIONAL_SPRINGS}')

## Joint angles during sweep

MCP and PIP angles for the four closed fingers (index / middle / ring / pinky)
during the sweep phase of each run.  All fingers should stay near
`FINGER_CLOSED_POSE` throughout; any large deviations indicate the spring was
insufficient to maintain the target pose against the string forces.

In [ ]:
fig, axes = plt.subplots(len(CLOSED_FINGERS), 1, figsize=(11, 8), sharex=True)
target_deg = np.rad2deg(FINGER_CLOSED_POSE[0])  # MCP target

for ax, (fname, midxs) in zip(axes, FINGER_MOTOR_IDX.items()):
    for i, k in enumerate(TORSIONAL_SPRINGS):
        d  = DATA[k]
        sw = d[d['phase'] == 'sweep'].copy()
        sw['time_s'] = sw['time_s'].astype(float)
        col = COLORS[i % len(COLORS)]
        for run in sorted(sw['run'].unique()):
            r = sw[sw['run'] == run]
            t = r['time_s'] - r['time_s'].min()
            q_mcp_deg = np.rad2deg(r[f'q_{midxs[0]}'].astype(float))
            ax.plot(t, q_mcp_deg, color=col, alpha=0.3, lw=1.2)
        # mean across runs
        runs = sorted(sw['run'].unique())
        tmax = min((sw[sw['run']==r]['time_s'].max() - sw[sw['run']==r]['time_s'].min())
                   for r in runs)
        grid = np.linspace(0, tmax, 200)
        interp = [np.interp(grid,
                            sw[sw['run']==r]['time_s'] - sw[sw['run']==r]['time_s'].min(),
                            np.rad2deg(sw[sw['run']==r][f'q_{midxs[0]}'].astype(float)))
                  for r in runs]
        ax.plot(grid, np.mean(interp, axis=0), color=col, lw=2.2,
                label=rf'$K={k:.1f}$ N$\cdot$m/rad')
    ax.axhline(target_deg, color='0.6', lw=1.0, ls='--', zorder=0)
    ax.set_ylabel(f'{fname}\nMCP [deg]')
    ax.set_ylim(bottom=0)

axes[0].legend(loc='upper right', fontsize='small', title='torsional stiffness')
axes[-1].set_xlabel('Time during sweep [s]')
if DEMO:
    fig.text(0.5, 0.5, 'DEMO', fontsize=90, color='0.92',
             ha='center', va='center', rotation=30, zorder=0)
fig.savefig(os.path.join(OUTPUT, 'guitar_joint_angles.pdf'))
plt.show()

## Applied torques during sweep

MCP torques for each finger during the sweep phase.  Higher torsional stiffness
produces larger restoring torques when the strings deflect the fingers.

In [ ]:
fig, axes = plt.subplots(len(CLOSED_FINGERS), 1, figsize=(11, 8), sharex=True)

for ax, (fname, midxs) in zip(axes, FINGER_MOTOR_IDX.items()):
    for i, k in enumerate(TORSIONAL_SPRINGS):
        d  = DATA[k]
        sw = d[d['phase'] == 'sweep'].copy()
        sw['time_s'] = sw['time_s'].astype(float)
        col = COLORS[i % len(COLORS)]
        for run in sorted(sw['run'].unique()):
            r = sw[sw['run'] == run]
            t = r['time_s'] - r['time_s'].min()
            ax.plot(t, r[f'tau_{midxs[0]}'].astype(float), color=col, alpha=0.3, lw=1.2)
        runs = sorted(sw['run'].unique())
        tmax = min((sw[sw['run']==r]['time_s'].max() - sw[sw['run']==r]['time_s'].min())
                   for r in runs)
        grid = np.linspace(0, tmax, 200)
        interp = [np.interp(grid,
                            sw[sw['run']==r]['time_s'] - sw[sw['run']==r]['time_s'].min(),
                            sw[sw['run']==r][f'tau_{midxs[0]}'].astype(float))
                  for r in runs]
        ax.plot(grid, np.mean(interp, axis=0), color=col, lw=2.2,
                label=rf'$K={k:.1f}$ N$\cdot$m/rad')
    ax.set_ylabel(f'{fname}\nMCP torque [N·m]')

axes[0].legend(loc='upper right', fontsize='small', title='torsional stiffness')
axes[-1].set_xlabel('Time during sweep [s]')
if DEMO:
    fig.text(0.5, 0.5, 'DEMO', fontsize=90, color='0.92',
             ha='center', va='center', rotation=30, zorder=0)
fig.savefig(os.path.join(OUTPUT, 'guitar_torques.pdf'))
plt.show()